## Configuration

In [1]:
import os
import duckdb
import pandas as pd
import json

from pathlib import Path

In [2]:
bac_info_file = "./data/bac_info.json"
per_prot_dir = "./data/inference_results/per_protein_results"
per_bac_dir = "./data/inference_results/per_bac_results"

os.makedirs(per_bac_dir, exist_ok=True)

## Bacterium-level Aggregation of PPI Predictions

* **Loads bacterial protein IDs** from a JSON file (`bac_info.json`) that maps each bacterium to its proteins.

* **Loops over each bacterium**:

  * Skips if an output file for that bacterium already exists.
  * Deduplicates its list of protein IDs.
  * Splits the list into manageable **chunks** (20,000 proteins at a time).
  * For each chunk:

    * Creates a small temporary table with the protein IDs.
    * Uses **DuckDB** to query all model inference results stored in parquet files.
    * **Joins** those results with the current chunk, keeping only rows where `bacterial_protein` matches and the prediction score is `1`.
    * Saves the matched interactions into a per-batch parquet file.

* In short: this script **collects model predictions for each bacterium’s proteins** by joining inference results with the bacterium’s protein list, and writes the matches out in smaller parquet files.

In [16]:
# Load bac_info.json
with open(bac_info_file, 'r') as f:
    bac_info = json.load(f)

# Connect DuckDB
con = duckdb.connect()
batch_size = 20000

for bac, info in bac_info.items():
    output_file = Path(per_bac_dir) / f"{bac}_interactions.parquet"

    if output_file.exists():
        print(f"{bac}: File already exists, skipping...")
        continue

    # Flatten and deduplicate protein list
    prot_list = list({prot for up_list in info.values() for prot in up_list})

    if not prot_list:
        print(f"{bac}: No proteins found, skipping...")
        continue

    for i in range(0, len(prot_list), batch_size):
        prot_chunk = prot_list[i:i + batch_size]
        prot_df = pd.DataFrame({"prot": prot_chunk})
        con.register("prot_table", prot_df)

        batch_file = Path(per_bac_dir) / f"{bac}_{i}_interactions.parquet"

        if batch_file.exists():
            print(f"{bac}: Batch file already exists, skipping...")
            continue

        query = """
            SELECT p.*
            FROM parquet_scan('./data/inference_results/per_protein_results/*.parquet') p
            INNER JOIN prot_table t
            ON p.bacterial_protein = t.prot
            WHERE p.score = 1
            """
        chunk_df = con.execute(query).df()

        if len(chunk_df) > 0:
            chunk_df.to_parquet(batch_file, index=False)
            print(f"Saved batch {i} for {bac} with {len(chunk_df)} rows")
        else:
            print(f"No interactions found for batch {i} for {bac}")



g__Alloscardovia: File already exists, skipping...
g__Arcanobacterium: File already exists, skipping...
g__Brevibacterium: File already exists, skipping...
g__CAG-352: File already exists, skipping...
g__Christensenellaceae R-7 group: File already exists, skipping...
g__Clostridium innocuum group: File already exists, skipping...
g__Colidextribacter: File already exists, skipping...
g__Corynebacterium: File already exists, skipping...
g__Cutibacterium: File already exists, skipping...
g__Dermabacter: File already exists, skipping...
g__Desulfovibrio: File already exists, skipping...
g__Dialister: File already exists, skipping...
g__Dorea: File already exists, skipping...
g_Shigella: File already exists, skipping...
g__Eubacterium hallii group: File already exists, skipping...
g__Eubacterium nodatum group: File already exists, skipping...
g__Facklamia: File already exists, skipping...
g__Family XIII AD3011 group: File already exists, skipping...
g__Filifactor: File already exists, skipp

**Total number of interactions**

In [19]:
query = """
    SELECT COUNT(*) AS total_records
    FROM parquet_scan('./data/inference_results/per_bac_results/*.parquet')
"""

result = con.execute(query).fetchone()[0]
print(f"Total number of interactions: {result}")

POTENTIAL_PAIRS = 8922520656
print(f"Fraction of potential pairs: {result}/{POTENTIAL_PAIRS} = {result / POTENTIAL_PAIRS:.2f}")

Total number of interactions: 1481068857
Fraction of potential pairs: 1481068857/8922520656 = 0.17


## Phenotype network contraction

* **Loads phenotype assignments** from `./data/bacterial_phenotype_assignment.csv`, keeping the `Taxonomy` (bacterium name) and `Phenotype_Assignment` columns.
* For each phenotype of interest (**“Healthy-associated”** and **“CRC-associated”**):

  * Filters the table to the bacteria labeled with that phenotype.
  * For each bacterium in that set, **finds all per-bacterium parquet files** in `per_bac_dir` whose filenames **start with the bacterium name**.
  * Opens each parquet file and extracts just the two columns: `bacterial_protein` and `human_protein`.
  * **Writes all pairs** to a phenotype-level text file:

    * `./data/inference_results/Healthy-associated_interactions.txt`
    * `./data/inference_results/CRC-associated_interactions.txt`
  * Lines are written as **tab-separated** `bacterial_protein<TAB>human_protein`.
  * Prints how many interactions were written per phenotype.

In short: it **contracts** the per-bacterium PPI results into **two phenotype-specific edge lists** (one for healthy-associated bacteria and one for CRC-associated), suitable for building phenotype-level PPI networks.

In [21]:
pheno_file = "./data/bacterial_phenotype_assignment.csv"

pheno_df = pd.read_csv(pheno_file)
pheno_df = pheno_df[["Taxonomy", "Phenotype_Assignment"]]

for ph in ["Healthy-associated", "CRC-associated"]:
    ph_df = pheno_df[pheno_df["Phenotype_Assignment"] == ph]

    pheno_bac_files = []

    for bac in ph_df["Taxonomy"].unique():
        # Take recursive glob all files starts with bac
        pheno_bac_files.extend(Path(per_bac_dir).glob(f"{bac}*.parquet"))
    
    ph_interactions_file = "./data/inference_results/{}_interactions.txt".format(ph)
    no_int = 0
    with open(ph_interactions_file, "w") as fout:
        for bac in pheno_bac_files:
            df = pd.read_parquet(bac, engine="pyarrow")

            # Vectorized write (faster than iterrows)
            lines = df[["bacterial_protein", "human_protein"]].astype(str).agg("\t".join, axis=1)
            fout.write("\n".join(lines) + "\n")

            no_int += len(df)

    print(f"{ph}: {no_int} interactions written to {ph_interactions_file}")

Healthy-associated: 553653445 interactions written to ./data/inference_results/Healthy-associated_interactions.txt
CRC-associated: 382499471 interactions written to ./data/inference_results/CRC-associated_interactions.txt


### Degree calculation per phenotype

In [ ]:
for ph in ["Healthy-associated", "CRC-associated"]:
    
    ph_interactions_file = "./data/inference_results/{}_interactions.txt".format(ph)

    hum_prot_degree = {}
    bac_prot_degree = {}

    with open(ph_interactions_file, "r") as f:
        for line in f:
            line = line.strip().split("\t")
            if len(line) != 2:
                continue

            hum_prot = line[1]
            bac_prot = line[0]

            if hum_prot not in hum_prot_degree:
                hum_prot_degree[hum_prot] = 1
            else:
                hum_prot_degree[hum_prot] += 1

            if bac_prot not in bac_prot_degree:
                bac_prot_degree[bac_prot] = 1
            else:
                bac_prot_degree[bac_prot] += 1

    # Save to csv
    hum_prot_degree_df = pd.DataFrame.from_dict(hum_prot_degree, orient="index", columns=["degree_human"])
    hum_prot_degree_df.sort_values("degree_human", ascending=False).reset_index().to_csv("./data/inference_results/{}_human_protein_degrees.csv".format(ph), index=False)

    bac_prot_degree_df = pd.DataFrame.from_dict(bac_prot_degree, orient="index", columns=["degree_bacterial"])
    bac_prot_degree_df.sort_values("degree_bacterial", ascending=False).reset_index().to_csv("./data/inference_results/{}_bacterial_protein_degrees.csv".format(ph), index=False)

Needed to do an id mapping in order to obtain higher degree proteins with non missing protein names and non missing GO: Either MF or BP. 

In [23]:
with pd.ExcelWriter("./data/inference_results/protein_degrees_by_phenotype.xlsx", engine="xlsxwriter") as writer:

    for ph in ["Healthy-associated", "CRC-associated"]:
        
        hum_degree_file = f"./data/{ph}_human_protein_degrees.csv"
        bac_degree_file = f"./data/{ph}_bacterial_protein_degrees.csv"

        hum_prot_degree = pd.read_csv(hum_degree_file, index_col=0)
        bac_prot_degree = pd.read_csv(bac_degree_file, index_col=0)

        bac_prot_degree = bac_prot_degree.reset_index()
        bac_prot_degree = bac_prot_degree.rename(columns={"index": "protein"})

        hum_prot_degree = hum_prot_degree.reset_index()
        hum_prot_degree = hum_prot_degree.rename(columns={"index": "protein"})

        bac_degree_mapping_file = f"./data/bacterial_{ph}_mapping.tsv.gz"
        hum_degree_mapping_file = f"./data/human_{ph}_mapping.tsv.gz"
        bac_degree_mapping = pd.read_csv(bac_degree_mapping_file, sep="\t", index_col=0) 
        # Drop nans of Protein names
        bac_degree_mapping = bac_degree_mapping.dropna(subset=["Protein names"])
        # Drop rows where both Gene Ontology (molecular function) and Gene Ontology (biological process) are NaN
        bac_degree_mapping = bac_degree_mapping.dropna(subset=["Gene Ontology (molecular function)", "Gene Ontology (biological process)"], how='all')

        # Merge the human/bacterial protein degree with the mapping. bac/hum_prot_degree protein column and bac/hum_degree_mapping Entry column. Keep columns from both. hum_degree_mapping had fiewer rows
        bac_prot_degree = bac_prot_degree.merge(bac_degree_mapping, left_on="protein", right_on="Entry", how="inner")
        bac_prot_degree = bac_prot_degree.drop(columns=["Entry"])

        # Same process
        hum_degree_mapping = pd.read_csv(hum_degree_mapping_file, sep="\t", index_col=0)
        hum_degree_mapping = hum_degree_mapping.dropna(subset=["Protein names"])
        hum_degree_mapping = hum_degree_mapping.dropna(subset=["Gene Ontology (molecular function)", "Gene Ontology (biological process)"], how='all')
        hum_prot_degree = hum_prot_degree.merge(hum_degree_mapping, left_on="protein", right_on="Entry", how="inner")
        hum_prot_degree = hum_prot_degree.drop(columns=["Entry"])

        # On both dataframes change. protein to Protein, and Degree_human or Degree_bacterial to Degree
        hum_prot_degree = hum_prot_degree.rename(columns={"protein": "Protein", "degree_human": "Degree"})
        bac_prot_degree = bac_prot_degree.rename(columns={"protein": "Protein", "degree_bacterial": "Degree"})
        
        print(bac_prot_degree.sort_values("Degree", ascending=False))
        print(hum_prot_degree.sort_values("Degree", ascending=False))
        print("-" * 100)

        hum_prot_degree.to_excel(writer, sheet_name=f"{ph}_Human", index=False)
        bac_prot_degree.to_excel(writer, sheet_name=f"{ph}_Bacterial", index=False)

        Protein  Degree                                      Protein names  \
0    A0A1T5GC38   29343            Cellulase (Glycosyl hydrolase family 5)   
1    A0A1T5D2V2   29307                Sugar phosphate isomerase/epimerase   
2    A0A1T5GAW1   29307                 Thymidylate synthase (EC 2.1.1.45)   
3    A0A1T5BRU8   29307                        ATPase/GTPase, AAA15 family   
4    A0A1T5CV36   29280                GTP cyclohydrolase 1 type 2 homolog   
..          ...     ...                                                ...   
389      R5DXX6   19532                        Pilus assembly protein CpaF   
390      R5I5H8   19530  Right handed beta helix domain-containing protein   
391      R6Q9D7   19530                 Dockerin domain-containing protein   
392      Q8A583   19530                Formamidopyrimidine-DNA glycosylase   
393      R6QCU0   19530  Ribulose-5-phosphate reductase (Ribulose-5-P r...   

                    Gene Ontology (biological process)  \
0    

In [10]:
# Read excel all sheets of this excel: "./data/inference_results/protein_degrees_by_phenotype.xlsx"
degree_file = "./data/inference_results/protein_degrees_by_phenotype.xlsx"
xls = pd.ExcelFile(degree_file)

with pd.ExcelWriter("./data/inference_results/central_protein_degrees_by_phenotype.xlsx", engine="xlsxwriter") as writer:

    for sheet_name in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet_name)
        
        # Mean
        mean = df["Degree"].mean()
        std = df["Degree"].std()
        max = df["Degree"].max()

        # Keep only mean + 2*std
        df = df[df["Degree"] <= mean + 2 * std]
        
        # To sheet
        df.to_excel(writer, sheet_name=sheet_name, index=False)